In [3]:
import pandas as pd
from Levenshtein import distance
from Levenshtein import ratio
from math import radians, cos, sin, asin, sqrt
from tqdm import tqdm
import re

# preprocess plaatsnamen
def preprocess_placename(place):
    if not isinstance(place, str):
        return place
    place = place.lower().strip()
    place = re.sub(r'^[^\w]+|[^\w]+$', '', place)
    return place

# Controleer of een string alleen uit cijfers en scheidingstekens bestaat
def is_numeric_with_separators(s):
    if not isinstance(s, str) or s == '':
        return False
    # Verwijder alle spaties en scheidingstekens (-, /, etc.)
    cleaned = re.sub(r'[\s\-\.\/,]', '', s)
    # Check of wat overblijft alleen cijfers zijn
    return cleaned.isdigit() and len(cleaned) > 0

# coördinaten Antwerpen
antwerp_lat, antwerp_lon = 51.2194, 4.4025

# haversine afstand
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # straal aarde in km
    return c * r

# toponiemen dictionary
toponyms = pd.read_csv('../../../data/H4/normalisering/toponyms_clean.tsv', sep='\t', dtype=str, na_values=['', ' ', 'NA'])
if 'latitude' in toponyms.columns:
    toponyms['latitude'] = toponyms['latitude'].apply(lambda x: float(x.replace(',', '.')) if isinstance(x, str) else x)
if 'longitude' in toponyms.columns:
    toponyms['longitude'] = toponyms['longitude'].apply(lambda x: float(x.replace(',', '.')) if isinstance(x, str) else x)

toponyms['latitude'] = pd.to_numeric(toponyms['latitude'], errors='coerce')
toponyms['longitude'] = pd.to_numeric(toponyms['longitude'], errors='coerce')

lookup = {}
for _, row in toponyms.iterrows():
    if isinstance(row['toponym'], str):
        key = preprocess_placename(row['toponym'])
        if key not in lookup:
            lookup[key] = []
        lookup[key].append(row.to_dict())

# voorbereiden gebeurtenisdata
incidents = pd.read_excel('../../../data/H4/normalisering/machinaal_test.xlsx')
incidents['P_BIRTHPLACE_extra'] = incidents['P_BIRTHPLACE'].str.extract(r'\((.*?)\)')
incidents['P_BIRTHPLACE_cleaned'] = incidents['P_BIRTHPLACE'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# fuzzy matching met geografische en tekstuele overeenkomst gebalanceerd
nearest_neighbors = []
for gemeente in tqdm(incidents['P_BIRTHPLACE_cleaned']):
    processed_gemeente = preprocess_placename(gemeente)

    # Check voor lege waarden
    if not isinstance(processed_gemeente, str) or processed_gemeente == '':
        nearest_neighbors.append([None, None, 'empty', None] + [None]*5)
        continue
    
    # Check voor numerieke waarden
    if processed_gemeente.isnumeric() or is_numeric_with_separators(processed_gemeente):
        nearest_neighbors.append([None, None, 'invalid', None] + [None]*5)
        continue

    #Levenshtein ratio
    distances = []
    for key in lookup.keys():
        similarity_ratio = ratio(key, processed_gemeente)
        if similarity_ratio >= 0.7:
            for entry in lookup[key]:
                distances.append((1.0 - similarity_ratio, similarity_ratio, key, entry))
    
    if not distances:
        nearest_neighbors.append([None, None, 'no_match_within_threshold', None] + [None]*5)
        continue
        
    sorted_distances = sorted(distances, key=lambda x: x[0])
    
    min_dist_equivalent = sorted_distances[0][0] if sorted_distances else float('inf')
    
    margin_matches = [d for d in sorted_distances if d[0] <= min_dist_equivalent + 0.15]
    
    #georanking van Lev matches
    haversine_distances = []
    for dist_equiv, sim_ratio, placename, entry in margin_matches:
        try:
            lat = entry['latitude']
            lon = entry['longitude']
            if pd.notna(lat) and pd.notna(lon):
                hav_dist = haversine(antwerp_lon, antwerp_lat, lon, lat)
                haversine_distances.append((dist_equiv, hav_dist, sim_ratio, placename, entry))
            else:
                haversine_distances.append((dist_equiv, float('inf'), sim_ratio, placename, entry))
        except (ValueError, TypeError):
            haversine_distances.append((dist_equiv, float('inf'), sim_ratio, placename, entry))
    
    exact_matches = any(d[1] == 1.0 for d in sorted_distances)
    
    if haversine_distances:
        haversine_distances.sort(key=lambda x: (x[1], x[0]))
        best_dist_equiv, best_hav_dist, best_sim_ratio, best_match_key, best_match_info = haversine_distances[0]
    else:
        best_dist_equiv, best_sim_ratio, best_match_key, best_match_info = sorted_distances[0]
        best_hav_dist = float('inf')
        match_type = 'fuzzy-fallback'
        
    consideration_pool_size = len(margin_matches)
    if best_sim_ratio >= 0.99:
        if consideration_pool_size > 1:
            match_type = 'exact+georanked'
        else:
            match_type = 'exact'
    else:
        exact_matches_exist = any(d[1] >= 0.99 for d in sorted_distances)
        if exact_matches_exist:
            match_type = 'geo-preferred'
        else:
            match_type = 'fuzzy'
    nearest_neighbors.append([
        best_sim_ratio,
        best_hav_dist, 
        match_type, 
        best_match_info['toponym'],
        best_match_info.get('pid', None),
        best_match_info.get('latitude', None),
        best_match_info.get('longitude', None),
        best_match_info.get('ISO', None),
        best_match_info.get('country', None)
    ])

result_df = pd.DataFrame(nearest_neighbors, columns=[
    'Similarity_Ratio', 'Haversine', 'Match Type', 
    'Best Match', 'pid', 'latitude', 'longitude', 'ISO', 
    'country'
])

selected_columns = result_df[['Best Match', 'country', 'Haversine', 'Similarity_Ratio']]
selected_columns.columns = ['P_BIRTHPLACE_match', 'P_BIRTHPLACE_country', 'P_BIRTHPLACE_dis', 'P_BIRTHPLACE_sim']

final_df = pd.concat([incidents, selected_columns], axis=1)
final_df.to_excel('../../../data/H4/normalisering/machinaal_test_birthplace.xlsx', index=False, header=True)

100%|██████████| 14179/14179 [13:27<00:00, 17.57it/s]
